# Recommendation System Demo

In [ ]:
import pandas as pd
import numpy as np
import joblib


In [ ]:
hybrid_model = joblib.load('../models/hybrid_stream_recommendation_model.pkl')
model_scaler = joblib.load('../models/model_scaler.pkl')
label_encoder = joblib.load('../models/stream_label_encoder.pkl')

kmeans_model = joblib.load('../models/kmeans_student_profile_model.pkl')
cluster_scaler = joblib.load('../models/cluster_scaler.pkl')


In [ ]:
career_paths = {
    'PCM': ['Engineering','Computer Science','Data Science'],
    'PCB': ['Medicine','Biotechnology','Pharmacy'],
    'Commerce with Maths': ['CA','Finance','Economics'],
    'Commerce without Maths': ['Business','Marketing','Management'],
    'Arts / Humanities': ['Law','Journalism','Psychology']
}


In [ ]:
sample_student = {
    'openness': 78,
    'conscientiousness': 82,
    'extraversion': 55,
    'agreeableness': 64,
    'neuroticism': 35,
    'numerical_aptitude': 88,
    'spatial_aptitude': 81,
    'perceptual_aptitude': 62,
    'abstract_reasoning': 85,
    'verbal_reasoning': 67
}

input_df = pd.DataFrame([sample_student])

input_df['stem_score'] = input_df[['numerical_aptitude','abstract_reasoning','spatial_aptitude']].mean(axis=1)
input_df['medical_score'] = input_df[['perceptual_aptitude','conscientiousness','agreeableness']].mean(axis=1)
input_df['commerce_score'] = input_df[['numerical_aptitude','verbal_reasoning','conscientiousness']].mean(axis=1)
input_df['arts_score'] = input_df[['verbal_reasoning','openness','agreeableness']].mean(axis=1)

cluster_input = input_df.select_dtypes(include=['int64','float64'])
cluster_scaled = cluster_scaler.transform(cluster_input)
input_df['student_cluster'] = kmeans_model.predict(cluster_scaled)

for col in hybrid_model.feature_names_in_:
    if col not in input_df.columns:
        input_df[col] = 0

input_df = input_df[hybrid_model.feature_names_in_]

scaled_input = model_scaler.transform(input_df)

probabilities = hybrid_model.predict_proba(scaled_input)[0]

indices = np.argsort(probabilities)[::-1][:3]

for idx in indices:
    stream = label_encoder.inverse_transform([idx])[0]
    confidence = round(probabilities[idx] * 100, 2)

    print(f'Stream: {stream}')
    print(f'Confidence: {confidence}%')
    print('Career Paths:', ', '.join(career_paths.get(stream, [])))
    print()